In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.model_selection import KFold

np.random.seed(42)
import random
random.seed(42)

#### Определяем функцию MAPE для оценки качества модели

In [2]:
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

#### Загружаем обучающую и тестовую выборки. Логарифмическое преобразование для стабилизации распределения

In [3]:
train_df = pd.read_csv('period_1_train_data.csv')
test_df = pd.read_csv('test_x.csv')

TARGET = 'price_target'
y = train_df[TARGET].values
y_log = np.log1p(y)  # Log transform target
train_df = train_df.drop(columns=[TARGET])

test_ids = test_df['id'].values if 'id' in test_df.columns else np.arange(len(test_df))
test_df = test_df.drop(columns=['id'])

print(f"\nTrain: {len(train_df)}, Test: {len(test_df)}")


Train: 29905, Test: 7477


#### Кодируем категориальные признаки с помощью LabelEncoder, обучая его на объединении train и test для согласованности кодирования

In [4]:
for col in train_df.columns:
    if train_df[col].dtype == 'object':
        le = LabelEncoder()
        combined = pd.concat([train_df[col].astype(str), test_df[col].astype(str)])
        le.fit(combined)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))

#### Заменяем -999 на пропуски и заполняем их медианными значениями по каждому признаку

In [5]:
train_df = train_df.replace(-999, np.nan)
test_df = test_df.replace(-999, np.nan)

for col in train_df.columns:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med)
    test_df[col] = test_df[col].fillna(med)

#### Преобразуем данные в числовой формат и формируем матрицы признаков

In [6]:
X = train_df.values.astype(np.float32)
X_test = test_df.values.astype(np.float32)

print(f"Features: {X.shape[1]}")

Features: 82


#### Кросс-валидация для оценки предсказания модели

In [7]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X)):
    model = xgb.XGBRegressor(
        n_estimators=8000,
        max_depth=11,
        learning_rate=0.009,
        subsample=0.72,
        colsample_bytree=0.68,
        min_child_weight=1,
        reg_alpha=0.04,
        reg_lambda=0.5,
        n_jobs=-1,
        verbosity=0,
        random_state=42
    )

    model.fit(
        X[tr_idx], y_log[tr_idx],
        eval_set=[(X[va_idx], y_log[va_idx])],
        verbose=False
    )

    oof[va_idx] = model.predict(X[va_idx])
    test_preds += model.predict(X_test) / 5

    oof_original = np.expm1(oof[va_idx])
    print(f"Fold {fold+1}: MAPE={mape(y[va_idx], oof_original):.4f}%")

test_preds_cv_original = np.expm1(test_preds)

submission_cv = pd.DataFrame({
    'id': test_ids,
    'price_target': test_preds_cv_original
})

submission_cv.to_csv('submission_cv.csv', index=False)
print("Saved submission_cv.csv")

Fold 1: MAPE=1.8599%
Fold 2: MAPE=1.8673%
Fold 3: MAPE=1.8688%
Fold 4: MAPE=1.9135%
Fold 5: MAPE=1.9425%
Saved submission_cv.csv


In [8]:
oof_original = np.expm1(oof)
print(f"\nFINAL OOF MAPE: {mape(y, oof_original):.4f}%")


FINAL OOF MAPE: 1.8904%


#### Финальная модель на всей обучающей выборке. Гиперпараметры подобраны перебором от результатов кросс-валидации

In [11]:
final_model = xgb.XGBRegressor(
    n_estimators=8000,
    max_depth=20,
    learning_rate=0.009,
    subsample=0.72,
    colsample_bytree=0.68,
    min_child_weight=1,
    reg_alpha=0.04,
    reg_lambda=0.5,
    n_jobs=-1,
    verbosity=0,
    random_state=42
)

final_model.fit(X, y_log, verbose=False)
test_preds_final = final_model.predict(X_test)

test_preds_original = np.expm1(test_preds_final)

submission_full = pd.DataFrame({
    'id': test_ids,
    'price_target': test_preds_original
})

submission_full.to_csv('submission.csv', index=False)
print("Saved submission.csv")

Saved submission.csv
